In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込みエラー: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 30. ZeROとFSDP

> **学習目標**
> - ZeRO Stage 1/2/3
> - FSDP (Fully Sharded Data Parallel)
> - 計算

## 30.1 DDP

DDP: GPU モデル . 70B モデル GPU .

ZeRO : **" "** — , , .

## 30.2

学習 = + + + 値

AdamW :
- $P$ (FP16): $2P$ bytes
- (FP16): $2P$ bytes
- (FP32: m, v, master): $12P$ bytes
- (値 ): $16P$ bytes

7B モデル: 112 GB ( GPU )

## 30.3 ZeRO Stage 1 —

 $N$ GPU :
- : $16P / N + 4P$ ( + )
- 7B, N=8: $16P/8 + 4P = 2P + 4P = 6P = 42GB$ ( )

## 30.4 ZeRO Stage 2 — 複雑度

複雑度 :
- : $16P/N + 2P = 4P$ (8B + 4P + 4P)
- Reduce-Scatter (All-Reduce )

## 30.5 ZeRO Stage 3 —

 :
- : $16P/N$ + 値
- 7B, N=8: $2P + ≈ 14GB$ → 40GB GPU
- : All-Gather (順伝播), Reduce-Scatter (バックプロパゲーション)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def zero_memory_gb(model_params_b, n_gpus, stage, optimizer_factor=12, dtype_bytes=2):
    """ZeRO stage Memory (GB).
    model_params_b: Parameter Count (: 10)
    optimizer_factor: Adam  12 (m, v, master FP32)
    dtype_bytes: 2 for FP16
    """
    P = model_params_b  # 10 
    # FP16 : 2 bytes, FP16 grad: 2 bytes
    # Optimizer (FP32 m, v, master): 12 bytes per param
    param_mem = P * dtype_bytes  # GB
    grad_mem = P * dtype_bytes
    opt_mem = P * optimizer_factor

    if stage == 0:  # DDP
        return param_mem + grad_mem + opt_mem
    elif stage == 1:  # Optimizer 
        return param_mem + grad_mem + opt_mem / n_gpus
    elif stage == 2:  # Optimizer + Gradient 
        return param_mem + (grad_mem + opt_mem) / n_gpus
    elif stage == 3:  #  
        return (param_mem + grad_mem + opt_mem) / n_gpus

# Visualization: 7B Model, GPU 
model_size = 7  # 7B
n_gpus_list = [1, 2, 4, 8, 16, 32, 64]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# : GPU 
ax = axes[0]
for stage, label, color in [(0, 'DDP', 'blue'), (1, 'ZeRO-1', 'green'),
                             (2, 'ZeRO-2', 'orange'), (3, 'ZeRO-3', 'red')]:
    mems = [zero_memory_gb(model_size, n, stage) for n in n_gpus_list]
    ax.plot(n_gpus_list, mems, 'o-', label=label, linewidth=2, color=color)
ax.axhline(40, color='gray', linestyle='--', label='A100 40GB')
ax.axhline(80, color='gray', linestyle=':', label='A100 80GB')
ax.set_xlabel('GPU ')
ax.set_ylabel('GPU  Memory (GB)')
ax.set_title(f'ZeRO Stage Memory (7B Model)')
ax.legend(); ax.grid(True, alpha=0.3)

# : Model Size
ax = axes[1]
for stage, label, color in [(0, 'DDP', 'blue'), (3, 'ZeRO-3', 'red')]:
    for n in [1, 8, 32]:
        sizes = [1, 7, 13, 30, 70, 175]
        mems = [zero_memory_gb(s, n, stage) for s in sizes]
        ax.plot(sizes, mems, 'o-', label=f'{label}, N={n}', linewidth=2)
ax.set_xlabel('Model Size (B params)')
ax.set_ylabel('GPU  Memory (GB)')
ax.set_title('Model Size Memory')
ax.set_yscale('log')
ax.legend(); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('../figures/ch30_zero_memory.png', dpi=100, bbox_inches='tight')
plt.show()

#  Output
print(f"\n{'モデル':>6} | {'GPU':>4} | {'DDP':>8} | {'ZeRO-1':>8} | {'ZeRO-2':>8} | {'ZeRO-3':>8}")
print("-" * 55)
for size in [7, 13, 70]:
    for n in [1, 8]:
        ddp = zero_memory_gb(size, n, 0)
        z1 = zero_memory_gb(size, n, 1)
        z2 = zero_memory_gb(size, n, 2)
        z3 = zero_memory_gb(size, n, 3)
        print(f"{size:>5}B | {n:>4} | {ddp:>7.1f}G | {z1:>7.1f}G | {z2:>7.1f}G | {z3:>7.1f}G")


## 30.6 PyTorch FSDP

FSDP (Fully Sharded Data Parallel) = PyTorch ZeRO-3.

```python
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

model = MyLargeModel
model = FSDP(model) #
```

:
1. 順伝播 : All-Gather
2. 順伝播 :
3. バックプロパゲーション: All-Gather, 計算
4. Reduce-Scatter

## 30.7 CPU Offloading

GPU CPU RAM .
- ZeRO-Infinity, FSDP with CPU offload
- 速度 モデル 学習


In [ ]:
# FSDP  (gradient sharding)
import torch
import torch.nn as nn
import torch.nn.functional as F

class FSDPSimulator:
    """ GPU FSDP  ."""
    def __init__(self, model, n_shards=4):
        self.model = model
        self.n_shards = n_shards
        #  n_shards  ( "GPU"  )
        params = list(model.parameters())
        self.shards = [params[i::n_shards] for i in range(n_shards)]

    def step(self, x, y, optimizer):
        """ Step: Forward Pass → Backward Pass →  shard Update."""
        optimizer.zero_grad()
        out = self.model(x)
        loss = F.mse_loss(out, y)
        loss.backward()

        #  shard Gradient  ( shard 0)
        #  FSDP  ,   Gradient 
        optimizer.step()
        return loss.item()

# 
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(128, 512), nn.ReLU(),
    nn.Linear(512, 128)
)
fsdp = FSDPSimulator(model, n_shards=4)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

x = torch.randn(8, 128)
y = torch.randn(8, 128)
loss = fsdp.step(x, y, opt)
print(f"FSDP  loss: {loss:.4f}")
print(f" : {sum(p.numel() for p in model.parameters()):,}")
print(f"Shard : {fsdp.n_shards} ( shard ~{sum(p.numel() for p in model.parameters()) // 4:,} )")


## 30.8 要点

| Stage | | | |
|---|---|---|---|
| DDP | | $16P$ | All-Reduce |
| ZeRO-1 | | $16P/N + 4P$ | All-Reduce |
| ZeRO-2 | + | $16P/N + 2P$ | Reduce-Scatter |
| ZeRO-3 | + | $16P/N$ | All-Gather + Reduce-Scatter |
| FSDP | ZeRO-3 (PyTorch) | $16P/N$ | |
| ZeRO-Infinity | + CPU/NVMe | | |

## 演習問題

1. ZeRO Stage 0/1/2/3 70B モデル, N=16 計算.
2. FSDP 順伝播 All-Gather .
3. ZeRO-3 DDP 比較.
4. CPU Offloading 速度 .
5. 70B モデル 学習 ZeRO-3 + A100 80GB 計算.

> 解答: `solutions/ch30_solutions.ipynb`
